In [1]:
import os
import json
import pandas as pd
import traceback
from dotenv import load_dotenv
from langchain_core.globals import set_debug

In [2]:
from langchain_openai import ChatOpenAI
load_dotenv()
set_debug(True)


In [3]:
KEY=os.getenv("GOOGLE_API_KEY")

In [4]:
llm = ChatOpenAI(api_key=KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",model="gemini-3.5-flash-lite",temperature=0.5)

In [5]:
llm

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x0000022C42A40BB0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000022C42F41240>, model_name='gemini-3.5-flash-lite', temperature=0.5, openai_api_key=SecretStr('**********'), openai_api_base='https://generativelanguage.googleapis.com/v1beta/openai/', openai_proxy='')

In [6]:
from langchain.llms import OpenAI
from langchain.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain.chains import SequentialChain
from langchain.callbacks import get_openai_callback
import PyPDF2

In [7]:
RESPONSE_JSON = {
    "1" : {
        "mcq": "multiple choice questions",
        "options": {
            "a" : "choice here",
            "b" : "choice here",
            "c" : "choice here",
            "d" : "choice here",
        },
        "correct": "correct answer",
    },
    "2" : {
        "mcq": "multiple choice questions",
        "options": {
            "a" : "choice here",
            "b" : "choice here",
            "c" : "choice here",
            "d" : "choice here",
        },
        "correct": "correct answer",
    },
    "3" : {
        "mcq": "multiple choice questions",
        "options": {
            "a" : "choice here",
            "b" : "choice here",
            "c" : "choice here",
            "d" : "choice here",
        },
        "correct": "correct answer",
    },
}

In [8]:
TEMPLATE="""
Text:{text}
You are an expert MCQ maker. Given the above text, it is your job to \
create a quiz of {number} multiple choice questions for {subject} students in {tone} tone.
Make sure the questions are not repeated and check all the questions to be conforming the text as well.
Make sure to format your response like RESPONSE_JSON below and use it as a guide. \
Ensure to make {number} MCQs
### RESPONSE_JSON
{response_json}

"""

In [9]:
quiz_generation_prompt = PromptTemplate(
    input_variables=["text","number","subject","tone","response_json"],
    template=TEMPLATE
)

In [10]:
quiz_chain = RunnablePassthrough.assign(
    quiz = quiz_generation_prompt | llm | StrOutputParser()
) 

In [11]:
TEMPLATE2="""
You are an expert english grammarian and writer. Given a Multiple Choice Quiz for {subject} students.\
You need to evaluate the complexity of the question and give a complete analysis of the quiz. Only use at max 50 words for complexity.
If the quiz is not at per with the cognitive and analytical abilities of the students,\
update the quiz questions which needs to be changed and change the tone such that it perfectly fits the students abilities \
Quiz_MCQs:
{quiz}

Chcek from an expert English Writer of the above quiz:
"""

In [12]:
quiz_evaluation_prompt = PromptTemplate(input_variables=["subject","quiz"], template = TEMPLATE2)

In [13]:
review_chain = RunnablePassthrough.assign(review = quiz_evaluation_prompt | llm | StrOutputParser())

In [14]:
generate_evaluate_chain = quiz_chain | review_chain

In [15]:
file_path=r"C:\Users\PRECISION\Desktop\Projects\mcqgen\data.txt"

In [16]:
with open(file_path, 'r') as file:
    TEXT = file.read()

In [17]:
print(TEXT)

The term machine learning was coined in 1959 by Arthur Samuel, an IBM employee and pioneer in the field of computer gaming and artificial intelligence.[6][7] The synonym self-teaching computers was also used during this time period.[8][9]

The earliest machine learning program was introduced in the 1950s, when Samuel invented a computer program that calculated the chance of winning in checkers for each side, but the history of machine learning is rooted in decades of efforts to study human cognitive processes.[10] In 1949, Canadian psychologist Donald Hebb published the book The Organization of Behavior, in which he introduced a theoretical neural structure formed by certain interactions among nerve cells.[11] The Hebbian theory of neuron interaction set the groundwork for how many machine learning algorithms work, with connected artificial neurons changing the strength of their connections based on data.[10] Other researchers who have studied human cognitive systems contributed to the

In [18]:
json.dumps(RESPONSE_JSON)

'{"1": {"mcq": "multiple choice questions", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "2": {"mcq": "multiple choice questions", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "3": {"mcq": "multiple choice questions", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}}'

In [19]:
NUMBER = 5
SUBJECT = "machine learning"
TONE="simple"

In [20]:
#How to get Token Usage Tracking in Langchain
with get_openai_callback() as cb:
    response=generate_evaluate_chain.invoke(
        {
            "text": TEXT,
            "number": NUMBER,
            "subject": SUBJECT,
            "tone": TONE,
            "response_json": json.dumps(RESPONSE_JSON)
        }
    )

   

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "text": "The term machine learning was coined in 1959 by Arthur Samuel, an IBM employee and pioneer in the field of computer gaming and artificial intelligence.[6][7] The synonym self-teaching computers was also used during this time period.[8][9]\n\nThe earliest machine learning program was introduced in the 1950s, when Samuel invented a computer program that calculated the chance of winning in checkers for each side, but the history of machine learning is rooted in decades of efforts to study human cognitive processes.[10] In 1949, Canadian psychologist Donald Hebb published the book The Organization of Behavior, in which he introduced a theoretical neural structure formed by certain interactions among nerve cells.[11] The Hebbian theory of neuron interaction set the groundwork for how many machine learning algorithms work, with connected artificial neurons changing the strength of their connections based on da

In [21]:
print(f"Total Tokens: {cb.total_tokens}")
print(f"Prompt Tokens: {cb.prompt_tokens}")
print(f"Completion Tokens: {cb.completion_tokens}")
print(f"Total Cost: {cb.total_cost}")

Total Tokens: 2898
Prompt Tokens: 1636
Completion Tokens: 1262
Total Cost: 0.0


In [22]:
response

{'text': 'The term machine learning was coined in 1959 by Arthur Samuel, an IBM employee and pioneer in the field of computer gaming and artificial intelligence.[6][7] The synonym self-teaching computers was also used during this time period.[8][9]\n\nThe earliest machine learning program was introduced in the 1950s, when Samuel invented a computer program that calculated the chance of winning in checkers for each side, but the history of machine learning is rooted in decades of efforts to study human cognitive processes.[10] In 1949, Canadian psychologist Donald Hebb published the book The Organization of Behavior, in which he introduced a theoretical neural structure formed by certain interactions among nerve cells.[11] The Hebbian theory of neuron interaction set the groundwork for how many machine learning algorithms work, with connected artificial neurons changing the strength of their connections based on data.[10] Other researchers who have studied human cognitive systems contri

In [23]:
quiz=response.get("quiz")

In [27]:
# 1. Grab the raw quiz string
raw_quiz = response["quiz"]

# 2. Clean the markdown code blocks (handles both ```json ... ``` and standard ``` ... ```)
cleaned_quiz = raw_quiz.strip()
if cleaned_quiz.startswith("```json"):
    cleaned_quiz = cleaned_quiz[7:]  # Remove ```json
elif cleaned_quiz.startswith("```"):
    cleaned_quiz = cleaned_quiz[3:]   # Remove ```

if cleaned_quiz.endswith("```"):
    cleaned_quiz = cleaned_quiz[:-3]  # Remove trailing ```

cleaned_quiz = cleaned_quiz.strip()

# 3. Now json.loads will work perfectly!
quiz_dict = json.loads(cleaned_quiz)
print(quiz_dict["1"]["mcq"])  # Successfully prints the question!

Who coined the term 'machine learning' in 1959?


In [28]:
quiz_table_data = []
for key, value in quiz_dict.items():
    mcq = value["mcq"]
    options = " | ".join(
        [
            f"{option}:{option_value}"
            for option, option_value in value["options"].items()
        ]
    )
    correct = value["correct"]
    quiz_table_data.append({"MCQ": mcq, "Choices": options, "Correct": correct})

In [29]:
quiz_table_data

[{'MCQ': "Who coined the term 'machine learning' in 1959?",
  'Choices': 'a:Donald Hebb | b:Arthur Samuel | c:Alan Turing | d:Tom M. Mitchell',
  'Correct': 'b'},
 {'MCQ': "What was the name of the early 'learning machine' developed by Raytheon Company in the early 1960s?",
  'Choices': 'a:Cybertron | b:AlphaGo | c:AlexNet | d:Word2vec',
  'Correct': 'a'},
 {'MCQ': "In Tom M. Mitchell's formal definition of machine learning, what do the letters E, T, and P stand for?",
  'Choices': 'a:Environment, Task, Processing | b:Experience, Task, Performance measure | c:Evaluation, Training, Program | d:Experiment, Theory, Prediction',
  'Correct': 'b'},
 {'MCQ': 'Which breakthrough computer program became the first to defeat a professional human Go player without handicaps in 2016?',
  'Choices': 'a:Cybertron | b:AlexNet | c:AlphaGo | d:Transformer',
  'Correct': 'c'},
 {'MCQ': 'What neural network architecture introduced in 2017 is based primarily on attention rather than recurrence or convolut

In [32]:
quiz=pd.DataFrame(quiz_table_data)

In [33]:
quiz.to_csv("machinelearning.csv",index=False)